# XGBoost Term Classification V2

This notebook trains XGBoost models using **MultiOutputClassifier** (much faster than V1!).

**Key improvements:**
- Uses `MultiOutputClassifier` instead of one-model-per-term
- ~10-20x faster training
- Supports hyperparameter tuning with `RandomizedSearchCV`
- PCA applied separately as preprocessing

**Sections:**
1. CogAtlas data (no PCA)
2. CogAtlas data (with PCA)
3. Hierarchical data (no PCA)
4. Hierarchical data (with PCA)
5. Hyperparameter tuning
6. Model comparison
7. Save/load models

In [ ]:
import sys
sys.path.append('../../')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

from neurovlm.term_classification_models import (
    load_and_align_term_data,
    XGBoostTermPredictor,
    evaluate_term_prediction,
    print_term_evaluation_results,
    save_term_results
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. CogAtlas Data (No PCA)

In [ ]:
# Load data
print("Loading CogAtlas term data...")
X, y, pmids, term_names = load_and_align_term_data(
    data_source="cogatlas"
)

print(f"\nDataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Number of terms: {len(term_names)}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

In [ ]:
# Train XGBoost (no PCA)
print("\nTraining XGBoost (no PCA)...")
xgb_model = XGBoostTermPredictor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

In [ ]:
# Evaluate
print("\nEvaluating...")
y_pred_proba = xgb_model.predict_proba(X_test)
y_pred = xgb_model.predict(X_test, threshold=0.5)

results_xgb_no_pca = evaluate_term_prediction(
    y_test, y_pred, y_pred_proba, term_names, top_k=10
)

print_term_evaluation_results(results_xgb_no_pca)

## 2. CogAtlas Data (With PCA)

In [ ]:
# Apply PCA
print("Applying PCA...")
pca = PCA(n_components=256, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")
print(f"Shape after PCA: {X_train_pca.shape}")

In [ ]:
# Train XGBoost with PCA
print("\nTraining XGBoost (with PCA)...")
xgb_model_pca = XGBoostTermPredictor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb_model_pca.fit(X_train_pca, y_train)

In [ ]:
# Evaluate
print("\nEvaluating...")
y_pred_proba_pca = xgb_model_pca.predict_proba(X_test_pca)
y_pred_pca = xgb_model_pca.predict(X_test_pca, threshold=0.5)

results_xgb_pca = evaluate_term_prediction(
    y_test, y_pred_pca, y_pred_proba_pca, term_names, top_k=10
)

print_term_evaluation_results(results_xgb_pca)

## 3. Hierarchical Data (Intermediate - No PCA)

In [ ]:
# Load hierarchical data
print("Loading intermediate hierarchical term data...")
X_hier, y_hier, pmids_hier, term_names_hier = load_and_align_term_data(
    data_source="intermediate_hierarchical_term"
)

print(f"\nHierarchical dataset shape: {X_hier.shape}")
print(f"Labels shape: {y_hier.shape}")
print(f"Number of terms: {len(term_names_hier)}")

In [ ]:
# Split
X_train_hier, X_test_hier, y_train_hier, y_test_hier = train_test_split(
    X_hier, y_hier, test_size=0.2, random_state=42
)

# Train
print("\nTraining XGBoost on hierarchical data (no PCA)...")
xgb_model_hier = XGBoostTermPredictor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb_model_hier.fit(X_train_hier, y_train_hier)

In [ ]:
# Evaluate
print("\nEvaluating...")
y_pred_proba_hier = xgb_model_hier.predict_proba(X_test_hier)
y_pred_hier = xgb_model_hier.predict(X_test_hier, threshold=0.5)

results_xgb_hier = evaluate_term_prediction(
    y_test_hier, y_pred_hier, y_pred_proba_hier, term_names_hier, top_k=10
)

print_term_evaluation_results(results_xgb_hier)

## 4. Hierarchical Data (With PCA)

In [ ]:
# Apply PCA to hierarchical data
print("Applying PCA to hierarchical data...")
pca_hier = PCA(n_components=256, random_state=42)
X_train_hier_pca = pca_hier.fit_transform(X_train_hier)
X_test_hier_pca = pca_hier.transform(X_test_hier)

print(f"Explained variance: {pca_hier.explained_variance_ratio_.sum():.2%}")

# Train
print("\nTraining XGBoost on hierarchical data (with PCA)...")
xgb_model_hier_pca = XGBoostTermPredictor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb_model_hier_pca.fit(X_train_hier_pca, y_train_hier)

In [ ]:
# Evaluate
print("\nEvaluating...")
y_pred_proba_hier_pca = xgb_model_hier_pca.predict_proba(X_test_hier_pca)
y_pred_hier_pca = xgb_model_hier_pca.predict(X_test_hier_pca, threshold=0.5)

results_xgb_hier_pca = evaluate_term_prediction(
    y_test_hier, y_pred_hier_pca, y_pred_proba_hier_pca, term_names_hier, top_k=10
)

print_term_evaluation_results(results_xgb_hier_pca)

## 5. Hyperparameter Tuning

Use `RandomizedSearchCV` to find best hyperparameters for CogAtlas data.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

print("Setting up hyperparameter search...")

# Define custom Recall@K scorer for ranking quality
def recall_at_k(y_true, y_pred_proba, k=10):
    """Calculate mean Recall@K across all samples.

    This metric optimizes for retrieval quality - getting as many
    true labels as possible into the top-K predictions.
    """
    recalls = []
    for i in range(len(y_true)):
        top_k_indices = np.argsort(y_pred_proba[i])[-k:]
        true_indices = np.where(y_true[i] == 1)[0]
        if len(true_indices) > 0:
            hits = np.sum(np.isin(top_k_indices, true_indices))
            recalls.append(hits / len(true_indices))
    return np.mean(recalls) if recalls else 0.0

# Create scorer that works with predict_proba
recall_at_10_scorer = make_scorer(
    recall_at_k,
    greater_is_better=True,
    needs_proba=True,
    k=10
)

print("Using Recall@10 as optimization metric (optimizes for top-10 retrieval quality)")

# Define parameter distributions
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0.5, 1.0, 2.0]
}

# Create search object with Recall@10 scoring
search = RandomizedSearchCV(
    XGBoostTermPredictor(random_state=42, n_jobs=1),  # n_jobs=1 for inner model
    param_dist,
    n_iter=20,  # Try 20 random combinations
    cv=3,       # 3-fold cross-validation
    scoring=recall_at_10_scorer,  # Optimizes for top-10 retrieval
    random_state=42,
    n_jobs=-1,  # Parallelize across CV folds
    verbose=2
)

print("\nStarting hyperparameter search (this may take a while)...")
search.fit(X_train, y_train)

In [ ]:
# Show best parameters
print("\n" + "="*70)
print("Best Parameters Found")
print("="*70)
for param, value in search.best_params_.items():
    print(f"{param}: {value}")

print(f"\nBest CV Score (Recall@10): {search.best_score_:.4f}")

In [ ]:
# Evaluate best model on test set
print("\nEvaluating best model on test set...")
best_model = search.best_estimator_

y_pred_proba_tuned = best_model.predict_proba(X_test)
y_pred_tuned = best_model.predict(X_test, threshold=0.5)

results_xgb_tuned = evaluate_term_prediction(
    y_test, y_pred_tuned, y_pred_proba_tuned, term_names, top_k=10
)

print_term_evaluation_results(results_xgb_tuned)

## 6. Model Comparison

In [ ]:
# Compare all models
comparison = pd.DataFrame([
    {
        'Model': 'XGBoost (No PCA)',
        'Micro F1': results_xgb_no_pca['micro']['f1'],
        'Macro F1': results_xgb_no_pca['macro']['f1'],
        'Recall@5': results_xgb_no_pca['recall_at_k']['recall@5'],
        'Recall@10': results_xgb_no_pca['recall_at_k']['recall@10'],
        'AUC Micro': results_xgb_no_pca['auc']['micro'],
    },
    {
        'Model': 'XGBoost (With PCA)',
        'Micro F1': results_xgb_pca['micro']['f1'],
        'Macro F1': results_xgb_pca['macro']['f1'],
        'Recall@5': results_xgb_pca['recall_at_k']['recall@5'],
        'Recall@10': results_xgb_pca['recall_at_k']['recall@10'],
        'AUC Micro': results_xgb_pca['auc']['micro'],
    },
    {
        'Model': 'XGBoost Tuned',
        'Micro F1': results_xgb_tuned['micro']['f1'],
        'Macro F1': results_xgb_tuned['macro']['f1'],
        'Recall@5': results_xgb_tuned['recall_at_k']['recall@5'],
        'Recall@10': results_xgb_tuned['recall_at_k']['recall@10'],
        'AUC Micro': results_xgb_tuned['auc']['micro'],
    },
    {
        'Model': 'XGBoost Hierarchical (No PCA)',
        'Micro F1': results_xgb_hier['micro']['f1'],
        'Macro F1': results_xgb_hier['macro']['f1'],
        'Recall@5': results_xgb_hier['recall_at_k']['recall@5'],
        'Recall@10': results_xgb_hier['recall_at_k']['recall@10'],
        'AUC Micro': results_xgb_hier['auc']['micro'],
    },
    {
        'Model': 'XGBoost Hierarchical (With PCA)',
        'Micro F1': results_xgb_hier_pca['micro']['f1'],
        'Macro F1': results_xgb_hier_pca['macro']['f1'],
        'Recall@5': results_xgb_hier_pca['recall_at_k']['recall@5'],
        'Recall@10': results_xgb_hier_pca['recall_at_k']['recall@10'],
        'AUC Micro': results_xgb_hier_pca['auc']['micro'],
    }
])

print("\n" + "="*70)
print("XGBoost Model Comparison")
print("="*70)
print(comparison.to_string(index=False))

# Find best model
best_idx = comparison['Micro F1'].idxmax()
print(f"\n✅ Best model: {comparison.loc[best_idx, 'Model']} (F1={comparison.loc[best_idx, 'Micro F1']:.4f})")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# F1 scores
comparison[['Model', 'Micro F1', 'Macro F1']].set_index('Model').plot(kind='bar', ax=axes[0])
axes[0].set_title('F1 Scores Comparison')
axes[0].set_ylabel('F1 Score')
axes[0].legend(['Micro F1', 'Macro F1'])
axes[0].tick_params(axis='x', rotation=45)

# Recall@K
comparison[['Model', 'Recall@5', 'Recall@10']].set_index('Model').plot(kind='bar', ax=axes[1])
axes[1].set_title('Recall@K Comparison')
axes[1].set_ylabel('Recall')
axes[1].legend(['Recall@5', 'Recall@10'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Save Models

In [ ]:
# Save best model
print("Saving models...")

# Save tuned model
best_model.save('xgboost_tuned_model.pkl')

# Save hierarchical model (often performs well)
xgb_model_hier.save('xgboost_hierarchical_model.pkl')

# Save PCA transformers if using PCA
import pickle
with open('pca_cogatlas.pkl', 'wb') as f:
    pickle.dump(pca, f)
    
with open('pca_hierarchical.pkl', 'wb') as f:
    pickle.dump(pca_hier, f)

print("\n✅ Models saved!")
print("  - xgboost_tuned_model.pkl")
print("  - xgboost_hierarchical_model.pkl")
print("  - pca_cogatlas.pkl")
print("  - pca_hierarchical.pkl")

In [ ]:
# Save evaluation results
save_term_results(results_xgb_tuned, 'xgboost_tuned_results.json')
save_term_results(results_xgb_hier, 'xgboost_hierarchical_results.json')

print("\n✅ Results saved!")

## Load and Use Saved Model

In [ ]:
# Example: Load and use saved model
from neurovlm.term_classification_models import TermPredictionModel

# Load model
loaded_model = TermPredictionModel.load('xgboost_tuned_model.pkl')

# Make predictions
y_pred_loaded = loaded_model.predict_proba(X_test)

# Get top-10 predictions for first test sample
top_k_indices, top_k_scores = loaded_model.predict_top_k(X_test[:1], k=10)

print("Top 10 predicted terms for first test sample:")
for i, (idx, score) in enumerate(zip(top_k_indices[0], top_k_scores[0])):
    print(f"{i+1}. {term_names[idx]}: {score:.3f}")

## Summary

**Key Findings:**
- XGBoost with MultiOutputClassifier is much faster than V1 (one-model-per-term)
- PCA can help or hurt depending on the dataset
- Hyperparameter tuning improves performance
- Hierarchical labels often perform better (less class imbalance)

**Recommendations:**
1. Start with hierarchical data (better class balance)
2. Try both with and without PCA
3. Use hyperparameter tuning for production models
4. Monitor Recall@K metrics for retrieval tasks